In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from sklearn.preprocessing import normalize
from scipy.spatial.distance import pdist, squareform

INPUT_FILE = Path("data/results_supcon_fixed/all_supcon_embeddings_fixed.pkl")
OUTPUT_ROOT = Path("data/results_rsm")
CONDITIONS = ["FB Prot Wrong", "FB Prot Corr", "FB Phys Wrong", "FB Phys Corr", "TB Prot Wrong", "TB Prot Corr", "TB Phys Wrong", "TB Phys Corr"]

METRICS = ["svm", "cosine", "euclidean", "correlation", "cka"]


def compute_prototypes(embeddings, labels):
    """计算每个条件的类中心（原型向量），形状 (8, feature_dim)"""
    unique_labels = sorted(np.unique(labels))
    prototypes = []
    for label in unique_labels:
        mask = (labels == label)
        # 聚合该条件下的所有试次
        prototypes.append(np.mean(embeddings[mask], axis=0))
    return np.array(prototypes)


def feature_space_linear_cka(X, Y):
    """
    计算 Linear CKA (Centered Kernel Alignment)
    X, Y: (n_samples, n_features)
    注意：为了计算 Gram 矩阵的相似性，通常需要 n_samples 一致。
    这里会对齐样本数量（截断到较小值）。
    """
    # 1. 对齐样本数量 (截断到最小值)
    min_n = min(X.shape[0], Y.shape[0])
    X = X[:min_n, :]
    Y = Y[:min_n, :]

    # 2. 中心化 (Centering) - 沿着样本维度
    X = X - X.mean(axis=0)
    Y = Y - Y.mean(axis=0)

    # 3. 计算 Linear CKA
    # 公式: ||Y^T X||_F^2 / (||X^T X||_F * ||Y^T Y||_F)
    # 利用 Frobenius norm 的性质，这比构建巨大的 Gram 矩阵 (NxN) 更快且省内存
    numerator = np.linalg.norm(Y.T @ X)**2
    denominator = np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y)

    return numerator / (denominator + 1e-8)


def compute_rsm_cka(embeddings, labels):
    """构建 CKA 距离矩阵 (8x8)，值 = 1 - CKA"""
    unique_labels = sorted(np.unique(labels))
    n_classes = len(unique_labels)
    rsm = np.zeros((n_classes, n_classes))

    # 提取每个条件的数据矩阵
    class_matrices = []
    for label in unique_labels:
        class_matrices.append(embeddings[labels == label])

    # 成对计算
    for i in range(n_classes):
        for j in range(i, n_classes):  # 计算上三角
            cka_val = feature_space_linear_cka(class_matrices[i], class_matrices[j])
            dist = 1 - cka_val
            rsm[i, j] = dist
            rsm[j, i] = dist

    return rsm


def compute_rsm_svm(embeddings, labels, n_splits=5):
    """构建 SVM 解码准确率矩阵 (8x8)"""
    unique_labels = sorted(np.unique(labels))
    n_classes = len(unique_labels)
    rsm = np.zeros((n_classes, n_classes))

    # tqdm leave=False 避免刷屏
    pbar = tqdm(total=n_classes * (n_classes - 1) // 2, desc="    SVM Computing", leave=False)

    for i in range(n_classes):
        for j in range(i + 1, n_classes):
            mask = np.isin(labels, [unique_labels[i], unique_labels[j]])
            X_pair = embeddings[mask]
            y_pair = labels[mask]

            clf = LinearSVC(dual="auto", random_state=42, max_iter=2000)
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
            scores = cross_val_score(clf, X_pair, y_pair, cv=cv, scoring='accuracy')

            acc = np.mean(scores)
            rsm[i, j] = acc
            rsm[j, i] = acc
            pbar.update(1)

    pbar.close()

    # 强制对角线为 0，符合 LLM 数据集的 RDM 格式
    np.fill_diagonal(rsm, 0.0)
    return rsm


def compute_rsm_simple(prototypes, metric):
    """基于原型向量计算距离 (Cosine, Euclidean, Correlation)"""
    if metric == 'cosine':
        # Cosine Distance = 1 - Cosine Similarity
        return cosine_distances(prototypes)

    elif metric == 'euclidean':
        # 先做 L2 Normalization，再算欧氏距离
        # 这样数值范围会比较小，且更关注方向差异
        norm_protos = normalize(prototypes, norm='l2', axis=1)
        return euclidean_distances(norm_protos)

    elif metric == 'correlation':
        # Correlation Distance = 1 - Pearson Correlation
        # np.corrcoef 计算的是相关系数矩阵
        corr_matrix = np.corrcoef(prototypes)
        return 1 - corr_matrix

    return None


def save_and_plot(rsm, metric_name, task_id):
    """保存 .npy 数据和 .png 热力图"""
    # 1. 创建目录结构: data/results_rsm/{metric}/{task_id}/
    save_dir = OUTPUT_ROOT / metric_name / task_id
    save_dir.mkdir(parents=True, exist_ok=True)

    # 2. 保存数据 (像 LLM 数据集一样命名为 avg_rdm.npy)
    np.save(save_dir / "avg_rdm.npy", rsm)

    # 3. 绘制并保存图片
    plt.figure(figsize=(10, 8))  # 稍微调大画布以便容纳数字
    sns.heatmap(
        rsm,
        xticklabels=CONDITIONS,
        yticklabels=CONDITIONS,
        cmap="viridis",
        annot=True,  # 开启数字标记
        fmt=".2f",  # 设置格式为2位小数
        annot_kws={"size": 9},  # 设置数字字体大小
        square=True)

    plt.title(f"{metric_name.upper()}: {task_id}", fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.savefig(save_dir / "heatmap.png", dpi=150)
    plt.close()

In [6]:
print(f"Loading embeddings from {INPUT_FILE}...")
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

with open(INPUT_FILE, 'rb') as f:
    data = pickle.load(f)

print(f"Found {len(data)} tasks (Periods x ROIs). Starting RSM generation...")

# 遍历每个任务 (例如: Belief_Period_H5_Left_PFC)
for task_id, content in data.items():
    print(f"\nProcessing Task: {task_id}")

    embeddings = content['embeddings']
    labels = content['labels']

    # 1. 计算原型向量 (Prototypes)
    prototypes = compute_prototypes(embeddings, labels)

    # 2. 计算五种 RSM

    # --- A. SVM (Decoding Accuracy) ---
    rsm_svm = compute_rsm_svm(embeddings, labels)
    save_and_plot(rsm_svm, "svm", task_id)

    # --- B. Cosine (Distance) ---
    rsm_cosine = compute_rsm_simple(prototypes, "cosine")
    save_and_plot(rsm_cosine, "cosine", task_id)

    # --- C. Euclidean (Distance, on Normalized Features) ---
    rsm_euclidean = compute_rsm_simple(prototypes, "euclidean")
    save_and_plot(rsm_euclidean, "euclidean", task_id)

    # --- D. Correlation (Distance) ---
    rsm_corr = compute_rsm_simple(prototypes, "correlation")
    save_and_plot(rsm_corr, "correlation", task_id)

    # --- E. CKA (1 - Linear CKA) ---
    rsm_cka = compute_rsm_cka(embeddings, labels)
    save_and_plot(rsm_cka, "cka", task_id)

print(f"\n{'='*40}")
print(f"All done! Results saved to {OUTPUT_ROOT}")
print(f"Structure: {OUTPUT_ROOT}/<metric>/<task_id>/avg_rdm.npy")

Loading embeddings from data/results_supcon_fixed/all_supcon_embeddings_fixed.pkl...
Found 8 tasks (Periods x ROIs). Starting RSM generation...

Processing Task: Belief_Period_H5_Left_PFC



Processing Task: Belief_Period_H5_Left_TPJ



Processing Task: Belief_Period_H5_Right_PFC



Processing Task: Belief_Period_H5_Right_TPJ



Processing Task: Perspective_Period_H5_Left_PFC



Processing Task: Perspective_Period_H5_Left_TPJ



Processing Task: Perspective_Period_H5_Right_PFC



Processing Task: Perspective_Period_H5_Right_TPJ



All done! Results saved to data/results_rsm
Structure: data/results_rsm/<metric>/<task_id>/avg_rdm.npy
